In [1]:
# Change working directory
from pathlib import Path
import os

path = Path(r"P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Maple")

os.chdir(path)

print(Path.cwd())

P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Maple


In [2]:
# Load python libraries
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

In [3]:
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

# Load pre-project report
pre_rpt = read_rpt_file("SSF_SDMP_Maple_pre.rpt")

# Extract data
pre = pre_rpt.node_flooding_summary.copy()

# Ensure Node is index
pre.index.name = "Node"

# Rename columns
pre = pre.rename(columns={
    "Hours_Flooded": "Pre Hours Flooded",
    "Maximum_Rate_CFS": "Pre Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Pre Total Flood Vol (MG)",
})

# Keep only needed columns
pre = pre[[
    "Pre Hours Flooded",
    "Pre Max Flood Rate (cfs)",
    "Pre Total Flood Vol (MG)"
]]

pre.head()

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG)
Node,,,
sQ2251A,2.04,5.93,0.013
sQ2251B,1.21,1.12,0.005
sQ2252,2.29,8.10,0.017
sQ2253,2.23,9.07,0.019
sQ2254,1.94,4.63,0.012


In [4]:
# Load post-project report
post_rpt = read_rpt_file("SSF_SDMP_Maple_imp1.rpt")

# Extract data
post = post_rpt.node_flooding_summary.copy()

# Ensure Node is index
post.index.name = "Node"

# Rename columns
post = post.rename(columns={
    "Hours_Flooded": "Post Hours Flooded",
    "Maximum_Rate_CFS": "Post Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Post Total Flood Vol (MG)",
})

# Keep only needed columns
post = post[[
    "Post Hours Flooded",
    "Post Max Flood Rate (cfs)",
    "Post Total Flood Vol (MG)"
]]

post.head()

,Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG)
Node,,,
sQ2251A,0.73,2.80,0.005
sQ2252,1.14,7.78,0.012
sQ2253,1.06,7.23,0.013
sQ2254,0.81,3.32,0.007
sQ2255,0.90,2.63,0.006


In [5]:
# Merge on index (Node)
comparison = pre.join(post, how="outer")

# Fill missing values
fill_cols = [
    "Pre Total Flood Vol (MG)",
    "Post Total Flood Vol (MG)",
    "Pre Hours Flooded",
    "Post Hours Flooded"
]

for col in fill_cols:
    comparison[col] = comparison[col].fillna(0)

# -----------------------------
# Reduction calculations
# -----------------------------

# Flood volume reduction
comparison["Total Flood Vol Reduction (MG)"] = (
    comparison["Pre Total Flood Vol (MG)"] - comparison["Post Total Flood Vol (MG)"]
)

comparison["Total Flood Vol Percent Reduction"] = np.where(
    comparison["Pre Total Flood Vol (MG)"] > 0,
    (comparison["Total Flood Vol Reduction (MG)"] / comparison["Pre Total Flood Vol (MG)"]) * 100,
    0
)

# Flood duration reduction
comparison["Hours Flooded Reduction"] = (
    comparison["Pre Hours Flooded"] - comparison["Post Hours Flooded"]
)

comparison["Hours Flooded Percent Reduction"] = np.where(
    comparison["Pre Hours Flooded"] > 0,
    (comparison["Hours Flooded Reduction"] / comparison["Pre Hours Flooded"]) * 100,
    0
)

# Round values
comparison["Total Flood Vol Reduction (MG)"] = comparison["Total Flood Vol Reduction (MG)"].round(3)
comparison["Total Flood Vol Percent Reduction"] = comparison["Total Flood Vol Percent Reduction"].round(1)

comparison["Hours Flooded Reduction"] = comparison["Hours Flooded Reduction"].round(2)
comparison["Hours Flooded Percent Reduction"] = comparison["Hours Flooded Percent Reduction"].round(1)

# Sort by severity
comparison = comparison.sort_values(
    by="Pre Total Flood Vol (MG)",
    ascending=False
)

comparison.head(20)

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG),Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG),Total Flood Vol Reduction (MG),Total Flood Vol Percent Reduction,Hours Flooded Reduction,Hours Flooded Percent Reduction
Node,,,,,,,,,,
sQ2253,2.23,9.07,0.019,1.06,7.23,0.013,0.006,31.6,1.17,52.5
sQ2252,2.29,8.10,0.017,1.14,7.78,0.012,0.005,29.4,1.15,50.2
sQ2261,2.20,3.32,0.014,1.10,3.31,0.011,0.003,21.4,1.10,50.0
sQ2251A,2.04,5.93,0.013,0.73,2.80,0.005,0.008,61.5,1.31,64.2
sQ2256,2.23,3.28,0.013,1.11,3.01,0.010,0.003,23.1,1.12,50.2
sQ2262,2.20,3.26,0.013,1.11,3.23,0.010,0.003,23.1,1.09,49.5
sQ2260,2.13,3.90,0.013,1.03,3.76,0.010,0.003,23.1,1.10,51.6
sQ2258,2.22,3.39,0.013,1.12,3.39,0.010,0.003,23.1,1.10,49.5
sQ2254,1.94,4.63,0.012,0.81,3.32,0.007,0.005,41.7,1.13,58.2


In [6]:
# Reset index so Node becomes a column
export_df = comparison.reset_index()

# Export to CSV
export_df.to_csv("flood_comparison.csv", index=False)

print("Exported: flood_comparison.csv")

Exported: flood_comparison.csv


In [7]:
# -----------------------------
# Entire network flood reduction
# -----------------------------

# --- Number of flooded nodes ---
pre_flooded_nodes = (comparison["Pre Hours Flooded"] > 0).sum()
post_flooded_nodes = (comparison["Post Hours Flooded"] > 0).sum()
flooded_nodes_reduction = pre_flooded_nodes - post_flooded_nodes
flooded_nodes_percent_reduction = float(np.where(
    pre_flooded_nodes > 0,
    (flooded_nodes_reduction / pre_flooded_nodes) * 100,
    0
))

# --- Average flood duration (flooded nodes only) ---
pre_avg_duration = comparison.loc[comparison["Pre Hours Flooded"] > 0, "Pre Hours Flooded"].mean()
post_avg_duration = comparison.loc[comparison["Post Hours Flooded"] > 0, "Post Hours Flooded"].mean()
pre_avg_duration = pre_avg_duration if not pd.isna(pre_avg_duration) else 0.0
post_avg_duration = post_avg_duration if not pd.isna(post_avg_duration) else 0.0
avg_duration_reduction = pre_avg_duration - post_avg_duration
avg_duration_percent_reduction = float(np.where(
    pre_avg_duration > 0,
    (avg_duration_reduction / pre_avg_duration) * 100,
    0
))

# --- Volume ---
network_pre_total = comparison["Pre Total Flood Vol (MG)"].sum()
network_post_total = comparison["Post Total Flood Vol (MG)"].sum()
network_reduction_mg = network_pre_total - network_post_total

network_percent_reduction = np.where(
    network_pre_total > 0,
    (network_reduction_mg / network_pre_total) * 100,
    0
)

# -----------------------------
# Print results
# -----------------------------

print("--- Flood Volume ---")
print(f"Pre-Project Network Total Flood Volume: {network_pre_total:.3f} MG")
print(f"Post-Project Network Total Flood Volume: {network_post_total:.3f} MG")
print(f"Network Flood Volume Reduction: {network_reduction_mg:.3f} MG")
print(f"Network Flood Volume Percent Reduction: {network_percent_reduction:.1f}%")

print("\n--- Flooded Nodes ---")
print(f"Pre-Project Number of Flooded Nodes: {pre_flooded_nodes}")
print(f"Post-Project Number of Flooded Nodes: {post_flooded_nodes}")
print(f"Flooded Nodes Reduction: {flooded_nodes_reduction}")
print(f"Flooded Nodes Percent Reduction: {flooded_nodes_percent_reduction:.1f}%")

print("\n--- Average Flood Duration (Flooded Nodes Only) ---")
print(f"Pre-Project Avg Flood Duration: {pre_avg_duration:.2f} hrs")
print(f"Post-Project Avg Flood Duration: {post_avg_duration:.2f} hrs")
print(f"Avg Flood Duration Reduction: {avg_duration_reduction:.2f} hrs")
print(f"Avg Flood Duration Percent Reduction: {avg_duration_percent_reduction:.1f}%")


--- Flood Volume ---
Pre-Project Network Total Flood Volume: 0.190 MG
Post-Project Network Total Flood Volume: 0.130 MG
Network Flood Volume Reduction: 0.060 MG
Network Flood Volume Percent Reduction: 31.6%

--- Flooded Nodes ---
Pre-Project Number of Flooded Nodes: 17
Post-Project Number of Flooded Nodes: 16
Flooded Nodes Reduction: 1
Flooded Nodes Percent Reduction: 5.9%

--- Average Flood Duration (Flooded Nodes Only) ---
Pre-Project Avg Flood Duration: 1.85 hrs
Post-Project Avg Flood Duration: 0.90 hrs
Avg Flood Duration Reduction: 0.95 hrs
Avg Flood Duration Percent Reduction: 51.6%


In [8]:
# Fix: ensure these are scalars (not numpy arrays)
network_percent_reduction = float(network_percent_reduction)
network_hours_percent_reduction = float(network_hours_percent_reduction)

# -----------------------------
# Create summary table
# -----------------------------
summary_df = pd.DataFrame({
    "Description": [
        "Pre-Project Number of Flooded Nodes",
        "Post-Project Number of Flooded Nodes",
        "Flooded Nodes Reduction",
        "Flooded Nodes Percent Reduction (%)",
        "Pre-Project Avg Flood Duration - Flooded Nodes Only (hrs)",
        "Post-Project Avg Flood Duration - Flooded Nodes Only (hrs)",
        "Avg Flood Duration Reduction (hrs)",
        "Avg Flood Duration Percent Reduction (%)",
        "Pre-Project Network Total Flood Volume (MG)",
        "Post-Project Network Total Flood Volume (MG)",
        "Network Flood Volume Reduction (MG)",
        "Network Flood Volume Percent Reduction (%)",
        "Pre-Project Total Hours Flooded (hrs)",
        "Post-Project Total Hours Flooded (hrs)",
        "Total Hours Flooded Reduction (hrs)",
        "Hours Flooded Percent Reduction (%)"
    ],
    "Value": [
        int(pre_flooded_nodes),
        int(post_flooded_nodes),
        int(flooded_nodes_reduction),
        f"{round(flooded_nodes_percent_reduction, 1)}%",
        round(pre_avg_duration, 2),
        round(post_avg_duration, 2),
        round(avg_duration_reduction, 2),
        f"{round(avg_duration_percent_reduction, 1)}%",
        round(network_pre_total, 3),
        round(network_post_total, 3),
        round(network_reduction_mg, 3),
        f"{round(network_percent_reduction, 1)}%",
        round(network_pre_hours, 2),
        round(network_post_hours, 2),
        round(network_hours_reduction, 2),
        f"{round(network_hours_percent_reduction, 1)}%"
    ]
})

# -----------------------------
# Export summary CSV
# -----------------------------
summary_df.to_csv("network_flood_summary.csv", index=False)

print("Exported: network_flood_summary.csv")


NameError: name 'network_hours_percent_reduction' is not defined